# Classifying CBIS-DDSM mass crops: scratch CNN versus frozen ImageNet VGG16

This research prototype compares a randomly initialised CNN with a classifier trained on frozen ImageNet VGG16 features. Both use the same patient-separated development and test partitions. VGG16 is itself a CNN; the comparison is between two CNN-based approaches with different architectures and pretraining, not CNNs versus a separate model family.

**Recorded run:** the uploaded notebook contains training and evaluation outputs from 28 September 2026 for `primary_seed42`. These original outputs are retained below. Newly added checking cells have not been executed against your local dataset here. The model files, full prediction CSVs and DICOM files were not included in this upload, so the saved outputs are evidence of the reported run, not an independent reproduction.

The prediction unit is a **mass lesion crop**, classified as benign (0) or malignant (1). `BENIGN_WITHOUT_CALLBACK` is mapped to benign. This is not whole-mammogram lesion detection or patient-level screening, and the scores are not calibrated clinical probabilities.

Keep this notebook in the existing project folder beside `project.py`, `prepare_cbis.py`, `prepared_data/` and `runs/`. The runtime modules are preserved to maintain the existing run's source hash. Read the review notes and source appendix before exporting.


## 1 Check your Jupyter environment

Use the environment where you installed `requirements.txt`. If imports fail, select the correct kernel before continuing. The project can use a CPU; no GPU is required.

In [ ]:
import sys
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from IPython.display import display, Markdown, Image as DisplayImage
import project as P
from prepare_cbis import build_manifest
print("Python:", sys.executable)
print("TensorFlow:", tf.__version__)
print("Devices:", tf.config.list_physical_devices())
print("Notebook folder:", Path.cwd())


## 2 Configure local paths

Set DOWNLOADS and IMAGE_ROOT for your machine. Obtain CBIS-DDSM images and metadata separately. The original local manifest and override CSV are not distributed. Read README.md before running.


In [ ]:
from pathlib import Path

DOWNLOADS = Path.home() / "Downloads"

TRAIN_CSV = DOWNLOADS / "mass_case_description_train_set.csv"
TEST_CSV = DOWNLOADS / "mass_case_description_test_set.csv"
IMAGE_ROOT = DOWNLOADS / "cbis_ddsm"

MANIFEST = Path("data/manifest.csv")
DATA_DIR = Path("prepared_data")
RUNS_DIR = Path("runs")
OVERRIDES = Path("data/crop_overrides_combined_20260928_183503.csv")

cfg = P.Config()
cfg.validate()


## 3 Load the crop manifest

Reuse `data/manifest.csv` when present. Otherwise, the adapter uses the combined overrides, exact paths and unique DICOM series matches. A series may contain both tissue and an ROI mask, so ambiguous matches must be reviewed using internal DICOM metadata and pixels. Successful path mapping alone is not visual verification.

The saved successful output contains **1,696 crops from 892 patients**: 1,318 development crops and 378 official test crops. The earlier failed mapping cell and repeated recovery cells have been removed from the main workflow; the dated recovery outcome is recorded in the review notes.


In [ ]:
if MANIFEST.exists():
    print("Using existing manifest:", MANIFEST)
else:
    build_manifest(TRAIN_CSV, TEST_CSV, IMAGE_ROOT, MANIFEST,
                   overrides=OVERRIDES if OVERRIDES.is_file() else None)
manifest = P.read_manifest(MANIFEST)
display(manifest.head())
print("Images:", len(manifest), "Patients:", manifest.patient_id.nunique())


## 4 Prepare images and check patient separation

The official test set is retained. Fold zero of a five-fold `StratifiedGroupKFold` on the official training set creates validation data, using split seed 42. All views and lesions from each patient remain in one partition. Duplicate source content or identical prepared pixels anywhere in the dataset cause an error instead of being silently removed.

Each grayscale crop is scaled independently to 8-bit, resized with its aspect ratio preserved, and padded to 224 × 224 pixels. The three model input channels repeat the grayscale image. CNN scaling occurs inside the model; VGG16 preprocessing occurs in the data pipeline. No augmentation is used in this baseline.

The recorded partition sizes are 1,054 training, 264 validation and 378 test crops. Patient counts within the benign and malignant rows **must not be added**: a patient can have crops in both classes. The additional split summary below counts distinct patients directly.


In [ ]:
if DATA_DIR.exists():
    status_path = DATA_DIR / "status.json"
    if not status_path.is_file():
        raise RuntimeError("Data folder exists without status.json; inspect it before proceeding.")
    status = json.loads(status_path.read_text())
    if status.get("status") != "prepared":
        raise RuntimeError("Previous preparation failed. Inspect its failure log first.")
    if status.get("manifest_sha256") != P.sha256(MANIFEST):
        raise RuntimeError("Manifest changed after preparation. Preserve existing data and investigate.")
    if status.get("image_size") != cfg.image_size or status.get("split_seed") != cfg.split_seed:
        raise RuntimeError("Configuration does not match the prepared dataset.")
    audit = pd.read_csv(DATA_DIR / "audit.csv")
else:
    audit = P.prepare_data(MANIFEST, DATA_DIR, cfg)
P.validate_separation(audit)
display(pd.read_csv(DATA_DIR / "counts.csv"))


In [ ]:
display(audit.groupby("split").agg(images=("image_id", "size"), patients=("patient_id", "nunique")))


## 5 Inspect development images

Check that these are lesion images, not binary masks, and that the source-to-label mapping is correct. Verify more than this small sample if there are variations in source format. Do not assess test performance while designing the model.

In [ ]:
examples = audit[audit["split"] == "train"].groupby("label", group_keys=False).head(3)
fig, axes = plt.subplots(len(examples), 2, figsize=(8, 3 * len(examples)), squeeze=False)
for pair, (_, row) in zip(axes, examples.iterrows()):
    source = P.read_grayscale(row.image_path)
    pair[0].imshow(source, cmap="gray")
    pair[0].set_title(f"Original: {'Malignant' if row.label else 'Benign'}")
    pair[1].imshow(Image.open(row.prepared_path), cmap="gray")
    pair[1].set_title("Prepared input")
    for ax in pair:
        ax.axis("off")
plt.tight_layout()
plt.show()

## 6 Record or display the data inspection

For an already verified dataset, display its existing dated inspection record without replacing it. For a new dataset, inspect the original and prepared development images and their metadata, then set `DATA_CHECKED = True` and enter what you actually reviewed. A sample inspection does not certify every image or constitute a clinical assessment.


In [ ]:
DATA_CHECKED = False
REVIEW_NOTE = ""  # Only enter checks you actually performed.
data_status = json.loads((DATA_DIR / "status.json").read_text())
if data_status.get("data_verified"):
    display({key: data_status.get(key) for key in
             ["data_verified", "verification_utc", "reviewer_note"]})
elif DATA_CHECKED:
    P.verify_data(DATA_DIR, REVIEW_NOTE)
else:
    raise RuntimeError("Inspect the images and labels, then complete the inspection record before Section 7.")


## 7 Create an experiment record

A new experiment ID creates a new folder and preserves older runs. This captures the configuration, source, split hashes and installed versions. The code will not train until the data inspection is recorded.

In [ ]:
RUN = RUNS_DIR.resolve() / cfg.experiment_id
if RUN.exists():
    saved_cfg, run_status = P.load_run(RUN)
    from dataclasses import asdict
    if asdict(saved_cfg) != asdict(cfg):
        raise RuntimeError("Current configuration differs from the saved experiment.")
    if Path(run_status["data_dir"]).resolve() != DATA_DIR.resolve():
        raise RuntimeError("The saved run refers to a different prepared dataset.")
    print("Existing run:", RUN)
    display(run_status)
else:
    RUN = P.new_run(DATA_DIR, RUNS_DIR, cfg)
    print("Created:", RUN)


### 7.1 Check saved evidence before continuing

This read-only check validates split separation and the bytes of prepared images against their recorded hashes. For completed runs it also checks saved model hashes and logs, and recomputes metrics from the complete prediction CSV. It does not rerun model inference or prove that externally supplied artifacts are authentic. Stop and investigate any mismatch; do not edit hashes to bypass it.


In [ ]:
def check_experiment_evidence(run, verify_pixels=True):
    """Check local saved evidence without retraining or rewriting results."""
    import hashlib
    run = Path(run)
    saved_cfg, status = P.load_run(run)
    parts = {s: pd.read_csv(Path(status["data_dir"]) / f"{s}.csv")
             for s in ["train", "val", "test"]}
    for name, frame in parts.items():
        if frame.empty or set(frame["split"]) != {name}:
            raise ValueError(f"Invalid split contents: {name}")
        if set(frame["label"]) != {0, 1}:
            raise ValueError(f"Both classes are required in {name}")
    joined = pd.concat(parts.values(), ignore_index=True)
    if joined.image_id.duplicated().any():
        raise ValueError("An image ID appears more than once across split files")
    P.validate_separation(joined)
    if verify_pixels:
        for row in joined.itertuples(index=False):
            with Image.open(row.prepared_path) as im:
                if im.mode != "L" or im.size != (saved_cfg.image_size, saved_cfg.image_size):
                    raise ValueError(f"Unexpected prepared image format: {row.image_id}")
                digest = hashlib.sha256(im.tobytes()).hexdigest()
            if digest != row.pixel_hash:
                raise ValueError(f"Prepared pixels changed: {row.image_id}")
    if status["stage"] in {"trained", "evaluated"}:
        timing = pd.read_csv(run / "training_summary.csv").set_index("model")
        if set(timing.index) != {"cnn", "vgg16"} or not timing.index.is_unique:
            raise ValueError("Need one training summary per model")
        for name in ["cnn", "vgg16"]:
            if P.sha256(run / name / "model.keras") != timing.loc[name, "model_sha256"]:
                raise ValueError(f"Saved model changed: {name}")
            log = pd.read_csv(run / name / "training_log.csv")
            if len(log) != int(timing.loc[name, "epochs"]):
                raise ValueError(f"Epoch count mismatch: {name}")
            if not np.array_equal(log.epoch.to_numpy(), np.arange(len(log))):
                raise ValueError(f"Nonconsecutive epoch log: {name}")
            if not np.isfinite(log[["loss", "val_loss"]].to_numpy()).all():
                raise ValueError(f"Nonfinite losses: {name}")
            if int(np.argmin(log.val_loss.to_numpy())) + 1 != int(timing.loc[name, "best_epoch"]):
                raise ValueError(f"Best checkpoint epoch mismatch: {name}")
    if status["stage"] == "evaluated":
        pred = pd.read_csv(run / "evaluation/test_predictions.csv")
        test = parts["test"]
        if pred.image_id.duplicated().any() or set(pred.image_id) != set(test.image_id):
            raise ValueError("Predictions do not cover the saved test split exactly")
        aligned = pred.set_index("image_id").loc[test.image_id]
        if not np.array_equal(aligned.label.to_numpy(), test.label.to_numpy()):
            raise ValueError("Prediction labels differ from test labels")
        if not np.array_equal(aligned.patient_id.to_numpy(), test.patient_id.to_numpy()):
            raise ValueError("Prediction patient IDs differ from the test split")
        measured = pd.read_csv(run / "evaluation/measured_results.csv").set_index("model")
        expected = {"cnn", "vgg16", "training_majority_reference"}
        if set(measured.index) != expected or not measured.index.is_unique:
            raise ValueError("Unexpected measured-results rows")
        majority = int(parts["train"].label.value_counts().idxmax())
        for name in sorted(expected):
            score = (np.full(len(test), majority) if name == "training_majority_reference"
                     else aligned[f"{name}_score"].to_numpy())
            recalculated = P.metric_values(test.label, score, saved_cfg.threshold)
            for metric, value in recalculated.items():
                if not np.isclose(float(measured.loc[name, metric]), value, rtol=1e-6, atol=1e-8):
                    raise ValueError(f"Metric mismatch: {name}, {metric}")
            for field, value in [("images", len(test)), ("patients", test.patient_id.nunique()),
                                 ("threshold", saved_cfg.threshold)]:
                if not np.isclose(float(measured.loc[name, field]), value):
                    raise ValueError(f"Result metadata mismatch: {name}, {field}")
        print("Saved test metrics agree with the full prediction CSV.")
    print("Evidence checks passed for stage:", status["stage"])
    return joined.groupby("split").agg(images=("image_id", "size"),
                                      patients=("patient_id", "nunique"))


In [ ]:
display(check_experiment_evidence(RUN))


## 8 Train or load the two fitted models

The scratch CNN has two convolution blocks (32 and 64 filters), global average pooling, a 128-unit dense layer, dropout 0.5 and a sigmoid output. All 27,841 parameters are trainable. VGG16 uses frozen ImageNet convolutional features with global average pooling and the same head design; 65,793 of its 14,780,481 parameters are trainable. VGG16's development features are cached once.

Both models use Adam at 0.0001, binary cross-entropy, training-derived balanced class weights, batch size 16, a 30-epoch limit and validation-loss early stopping with patience 5. The lowest validation-loss checkpoint is restored. Dataset shuffling is implemented inside the input pipelines; Keras's saved `shuffle=True` warning does not mean that the training data were unshuffled.

The existing output records 9 CNN epochs (best epoch 4) and 18 VGG16 head epochs (best epoch 13). Re-running an evaluated run loads its summary instead of retraining. Interrupted runs must be preserved and investigated; automatic resume is not implemented. These are two specific pipelines, not an isolated causal test of pretraining alone.


In [ ]:
run_status = json.loads((RUN / "status.json").read_text())
if run_status["stage"] == "created":
    display(P.train_models(RUN))
elif run_status["stage"] in {"trained", "evaluated"}:
    display(pd.read_csv(RUN / "training_summary.csv"))
else:
    raise RuntimeError("An earlier training attempt did not complete. Preserve it and use a new ID.")

## 9 Inspect recorded training logs and timings

Training loss is class-weighted, while validation loss is unweighted; dropout is active during training. Their values are therefore not directly equivalent. Validation loss selects the saved checkpoint. CSV `epoch` values start at zero; `best_epoch` and plotted epochs start at one.

VGG16's fit time measures only classifier-head training. Compare total time as well as feature extraction and fit times: total time also includes model setup, weight download when needed, caching and saving. The saved run used a CPU. UTC timestamps are eight hours behind Singapore time. Keras's threshold-approximated training AUC can differ from the rank-based ROC-AUC calculated during evaluation.


In [ ]:
for name in ["cnn", "vgg16"]:
    print(name)
    display(pd.read_csv(RUN / name / "training_log.csv").tail())
display(pd.read_csv(RUN / "training_summary.csv"))

## 10 Evaluate the frozen experiment

Run after both models are trained. The fixed decision threshold is 0.5. This computes image-level test metrics, paired patient-cluster bootstrap intervals and figures. Do not use this test set repeatedly to choose architecture or hyperparameters. Reading existing results does not retrain or reevaluate.

In [ ]:
run_status = json.loads((RUN / "status.json").read_text())
if run_status["stage"] == "trained":
    results = P.evaluate_models(RUN)
elif run_status["stage"] == "evaluated":
    results = pd.read_csv(RUN / "evaluation/measured_results.csv")
else:
    raise RuntimeError("Complete both training runs first.")
display(results)

## 11 Compare uncertainty and figures

The difference interval is VGG16 minus CNN. An interval spanning zero does not support a clear difference at this resolution. The bootstrap captures variability of test patients for these fitted models; it does not capture retraining variability or prove clinical usefulness.

In [ ]:
display(pd.read_csv(RUN / "evaluation/paired_patient_bootstrap.csv"))
from IPython.display import Image as DisplayImage, display, Markdown
for filename in ["cnn_learning_curves.png", "vgg16_learning_curves.png",
                 "confusion_matrices.png", "roc_and_precision_recall.png"]:
    display(DisplayImage(filename=str(RUN / "evaluation" / filename)))
display(Markdown((RUN / "evaluation/RESULTS.md").read_text()))

## 12 Inspect disagreements and false negatives

Use both successful and unsuccessful cases. A missed malignant crop should be discussed as a model error, not hidden. These columns support an error analysis; they do not explain the model's reasoning.

In [ ]:
predictions = pd.read_csv(RUN / "evaluation/test_predictions.csv")
false_negatives = predictions[(predictions.label == 1) &
    ((predictions.cnn_score < 0.5) | (predictions.vgg16_score < 0.5))]
display(false_negatives.head(10))
disagreements = predictions[(predictions.cnn_score >= .5) != (predictions.vgg16_score >= .5)]
display(disagreements.head(10))
subgroups = RUN / "evaluation/exploratory_subgroups.csv"
if subgroups.exists():
    display(pd.read_csv(subgroups))

## 13 Demonstrate inference on an existing test crop

This cell selects a real file from the manifest and applies the same preprocessing used during training. Both saved models must exist. The selected image is already part of the test set: this is an inference demonstration, not a new independent evaluation. The saved example is malignant; the CNN predicted benign and VGG16 predicted malignant.

For another crop, replace `NEW_IMAGE` with `Path(r"C:\full\actual\image.dcm")` pointing to an existing tissue image. A sigmoid score is not automatically a calibrated disease probability.


In [ ]:
from pathlib import Path

NEW_IMAGE = Path(
    manifest.loc[
        manifest["official_split"] == "test", "image_path"
    ].iloc[0]
)

print("Image:", NEW_IMAGE)
assert NEW_IMAGE.is_file(), f"Image not found: {NEW_IMAGE}"

for model_name in ["cnn", "vgg16"]:
    print(model_name, P.predict_image(RUN, model_name, NEW_IMAGE))

## 14 Summarise actual stakeholder feedback

Use `stakeholders/feedback_form.md` to collect genuine, consented responses and record resulting changes in `stakeholders/change_log.csv`. The saved output reports **no stakeholder responses collected**. This aspect of the project remains incomplete. Peer feedback can inform usability and clarity, but cannot establish diagnostic performance or clinical usefulness.


In [ ]:
summary = P.summarise_feedback("stakeholders/responses.csv", "stakeholders/SUMMARY.md")
display(Markdown(summary))

## 15 Interpret the recorded comparison

The following values are transcribed from the original notebook's saved evaluation output, not generated by this review. The test set contains 378 crops from 201 patients, including 147 malignant and 231 benign crops. The decision threshold was fixed at 0.5.

| Model | Accuracy | ROC-AUC | Sensitivity | Specificity | F1 | False negatives |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| Scratch CNN | 0.6164 | 0.5359 | 0.0340 | 0.9870 | 0.0645 | 142 / 147 |
| Frozen VGG16 + trained head | 0.6032 | 0.6342 | 0.5374 | 0.6450 | 0.5130 | 68 / 147 |
| Training-majority reference | 0.6111 | 0.5000 | 0.0000 | 1.0000 | 0.0000 | 147 / 147 |

The CNN classified nearly every crop as benign: it detected only 5 malignant crops and produced 3 false positives. Its accuracy was only about 0.53 percentage points above the majority reference. Its training loss remained near 0.693, and displayed scores clustered near 0.5, indicating weak discrimination in this run. These observations do not establish the cause; architecture, optimisation and preprocessing could be investigated using development data.

VGG16 detected 79 malignant crops, with 82 false positives. It had higher ROC-AUC, sensitivity and F1, but lower specificity and overall accuracy. The saved paired patient-cluster bootstrap gives a VGG16-minus-CNN ROC-AUC difference interval of **0.0327 to 0.1756**, and a sensitivity difference interval of **0.4195 to 0.5878** (1,000 resamples). Both intervals exclude zero for this fitted-model comparison. They do not include uncertainty from retraining, other datasets or protocol choices.

Total recorded training-pipeline time was approximately 202.5 seconds for the CNN and 312.9 seconds for VGG16. The VGG16 head fit itself took about 6.8 seconds, but feature extraction took about 216.6 seconds; reporting head fit alone would misrepresent its overall cost.

The comparison objective has recorded experimental evidence, but this is one seed on annotated mass crops from one dataset. Crops assume a lesion has already been located. Resizing and intensity scaling can discard information; there is no external validation, probability calibration study or clinical evaluation. Stakeholder feedback remains uncollected. Do not tune the threshold or choose a replacement model using these test results and then call the same test set independent.

Use the saved run records to write your own reflection on decisions and limitations. Keep genuine planning dates and feedback separate from these measured outputs.


## Review record — 28 September 2026

- Removed obsolete failed mapping output and duplicate recovery cells; retained the successful manifest output. The recovery output recorded 815 additional mappings and 1,593 combined overrides in `crop_overrides_combined_20260928_183503.csv`.
- Corrected the title and descriptions to mass-crop classification, frozen feature extraction, patient grouping, weighted-loss interpretation and timing scope.
- Preserved original training, evaluation, inference and feedback outputs. Historical outputs remain labelled as recorded evidence; edited setup/check cells need to be run locally. No model was retrained and no result was substituted.
- Added configuration/manifest checks, direct unique-patient counts, saved-model and prepared-pixel integrity checks, metric recomputation from the complete prediction CSV, and an appendix/runtime comparison.
- The uploaded inspection cell was still set to `False`, although a later run was created. The revised cell displays the existing verification record when available rather than inventing a retrospective inspection note.
- Runtime module listings remain unchanged for compatibility with the source hash recorded by the fitted run. The original adapter still requires explicit overrides for ambiguous crop/mask series.
- Local review checked Python syntax, source-listing equality, core preprocessing, patient separation, metrics and the new evidence-check helper using synthetic fixtures. Real DICOM conversion, TensorFlow training/inference, and full output reproduction require your local files. The notebook alone cannot independently establish model-weight provenance or reproduce ROC-AUC/bootstrap intervals.

This review record is dated now; it is not a replacement for a contemporaneous project planning log or stakeholder feedback.


## 16 Full source-code appendix and provenance

Appendices A and B contain the complete module listings embedded in the uploaded notebook. They were syntax-checked in this review and preserved unchanged so that replacing the runtime source does not invalidate `primary_seed42`. They are reference listings; normal execution imports the `.py` files in the project folder.

The appendix does not silently claim to be the exact source on your computer. Run the comparison cell below after saving this notebook: it checks both runtime files against the listings. For the recorded training run, `P.load_run` separately checks the source hash saved at run creation. Investigate any mismatch rather than rewriting the saved hash.

Appendix C duplicates the new notebook verification helper exactly. Its checks add audit coverage without changing either fitted model or the underlying experimental protocol.


In [ ]:
# Update this filename if you rename the downloaded notebook.
NOTEBOOK_FILE = Path("CNN_VGG16_Comparison.ipynb")
notebook_content = json.loads(NOTEBOOK_FILE.read_text(encoding="utf-8"))
for heading, runtime in [
    ("### Appendix A — project.py", Path(P.__file__)),
    ("### Appendix B — prepare_cbis.py", Path("prepare_cbis.py"))
]:
    matches = ["".join(c["source"]) for c in notebook_content["cells"]
               if c["cell_type"] == "markdown" and "".join(c["source"]).startswith(heading)]
    if len(matches) != 1:
        raise ValueError(f"Missing or duplicated appendix: {heading}")
    listing = matches[0].split("```python\n", 1)[1].rsplit("```", 1)[0]
    if listing.replace("\r\n", "\n").strip() != runtime.read_text(encoding="utf-8").strip():
        raise ValueError(f"Appendix differs from runtime file: {runtime}")
    print("Appendix matches:", runtime.name)


### Appendix A — project.py (recorded baseline implementation)

```python
"""Patient-separated CNN versus frozen ImageNet VGG16 experiment.

No fabricated results: training/evaluation outputs are created only by actual runs.
TensorFlow is imported lazily so data and metric tests can run independently.
"""
from pathlib import Path
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
import hashlib
import csv
import importlib.metadata
import json
import platform
import re
import time
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight

LABELS = {"BENIGN": 0, "BENIGN_WITHOUT_CALLBACK": 0, "MALIGNANT": 1}


def utc_now():
    return datetime.now(timezone.utc).isoformat()


def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")


def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


@dataclass
class Config:
    image_size: int = 224
    batch_size: int = 16
    epochs: int = 30
    patience: int = 5
    learning_rate: float = 0.0001
    split_seed: int = 42
    train_seed: int = 42
    bootstrap_repetitions: int = 1000
    threshold: float = 0.5
    # Choose a NEW ID for a different training seed before viewing any test results.
    experiment_id: str = "primary_seed42"

    def validate(self):
        if self.image_size != 224 or self.threshold != 0.5:
            raise ValueError("Primary protocol fixes 224 pixels and threshold 0.5.")
        if min(self.batch_size, self.epochs, self.patience,
               self.bootstrap_repetitions) < 1 or self.learning_rate <= 0:
            raise ValueError("Invalid training or bootstrap configuration")
        if not re.fullmatch(r"[A-Za-z0-9_-]+", self.experiment_id):
            raise ValueError("Use letters/numbers/underscore/hyphen in experiment ID")


def canonical_patient(value):
    value = str(value).strip()
    match = re.search(r"P_\d+", value)
    return match.group(0) if match else value


def read_manifest(path):
    """Relative image paths are resolved relative to the manifest CSV directory."""
    path = Path(path).resolve()
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    required = {"patient_id", "image_path", "pathology", "official_split"}
    if not required.issubset(df.columns) or df.empty:
        raise ValueError(f"Need nonempty manifest with columns {sorted(required)}")
    for col in required:
        df[col] = df[col].str.strip()
        if (df[col] == "").any():
            raise ValueError(f"Blank {col} in manifest")
    df["patient_id"] = df.patient_id.map(canonical_patient)
    df["pathology"] = df.pathology.str.upper()
    df["official_split"] = df.official_split.str.lower()
    if not set(df.pathology).issubset(LABELS):
        raise ValueError("Unknown pathology; review labels instead of guessing")
    if set(df.official_split) != {"train", "test"}:
        raise ValueError("official_split must contain both train and test")
    def resolve(p):
        p = Path(p)
        return str((p if p.is_absolute() else path.parent / p).resolve())
    df["image_path"] = df.image_path.map(resolve)
    missing = df.loc[~df.image_path.map(lambda p: Path(p).is_file()), "image_path"]
    if len(missing):
        raise FileNotFoundError(f"Missing {len(missing)} images; first: {missing.iloc[0]}")
    if df.image_path.duplicated().any():
        raise ValueError("Repeated file path; audit duplicate or conflicting lesions")
    df["label"] = df.pathology.map(LABELS).astype(int)
    # Stable within a project folder; retain image IDs in saved manifests.
    if "image_id" not in df:
        df["image_id"] = [hashlib.sha256((p + '|' + s).encode()).hexdigest()[:20]
                          for p, s in zip(df.patient_id, df.image_path)]
    if (df.image_id == "").any() or df.image_id.duplicated().any():
        raise ValueError("image_id must be nonblank and unique")
    return df.reset_index(drop=True)


def validate_separation(df, column="split"):
    if df.groupby("patient_id")[column].nunique().max() > 1:
        raise ValueError("Patient leakage: a patient occurs in more than one split")
    if "pixel_hash" in df:
        duplicates = df[df.pixel_hash.duplicated(keep=False)]
        if not duplicates.empty:
            raise ValueError("Identical prepared pixels found; audit duplicates before training")
    if "raw_hash" in df and df.raw_hash.duplicated().any():
        raise ValueError("Identical source file content found; audit duplicates")


def make_split(df, seed=42):
    """Keep official test; take fold zero of a fixed five-fold development split."""
    validate_separation(df, "official_split")
    dev = df[df.official_split == "train"].copy()
    test = df[df.official_split == "test"].copy()
    if dev.patient_id.nunique() < 5 or dev.groupby("label").patient_id.nunique().min() < 5:
        raise ValueError("Need at least five development patients per class")
    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    ti, vi = next(splitter.split(dev, dev.label, groups=dev.patient_id))
    train, val = dev.iloc[ti].copy(), dev.iloc[vi].copy()
    train["split"], val["split"], test["split"] = "train", "val", "test"
    result = pd.concat([train, val, test], ignore_index=True)
    validate_separation(result)
    for name, part in result.groupby("split"):
        if set(part.label) != {0, 1}:
            raise ValueError(f"{name} lacks both classes. Review protocol before training.")
    return result


def read_grayscale(path):
    """Return float32 grayscale pixels; explicit DICOM MONOCHROME handling."""
    path = Path(path)
    if path.suffix.lower() in {".dcm", ".dicom"}:
        import pydicom
        from pydicom.pixels import apply_modality_lut
        ds = pydicom.dcmread(path)
        if str(getattr(ds, "PhotometricInterpretation", "")) not in {
                "MONOCHROME1", "MONOCHROME2"}:
            raise ValueError(f"Unsupported DICOM photometric interpretation: {path}")
        pixels = np.asarray(apply_modality_lut(ds.pixel_array, ds), dtype=np.float32)
        if pixels.ndim != 2:
            raise ValueError(f"Expected a single grayscale frame: {path}")
        if ds.PhotometricInterpretation == "MONOCHROME1":
            pixels = pixels.max() + pixels.min() - pixels
    else:
        with Image.open(path) as im:
            if im.mode in {"L", "I", "F", "I;16", "I;16B", "I;16L"}:
                pixels = np.array(im, dtype=np.float32)
            elif im.mode in {"RGB", "RGBA"}:
                rgb = np.asarray(im.convert("RGB"))
                if not (np.array_equal(rgb[..., 0], rgb[..., 1]) and
                        np.array_equal(rgb[..., 1], rgb[..., 2])):
                    raise ValueError(f"Image is not grayscale; check annotations: {path}")
                pixels = rgb[..., 0].astype(np.float32)
            else:
                raise ValueError(f"Unsupported image mode {im.mode}: {path}")
    if not np.isfinite(pixels).all() or min(pixels.shape) < 8:
        raise ValueError(f"Invalid image pixels: {path}")
    if np.unique(pixels).size <= 4:
        raise ValueError(f"Possible ROI mask or blank image, not a mammogram crop: {path}")
    return pixels


def prepare_image(path, size=224):
    pixels = read_grayscale(path)
    lo, hi = float(pixels.min()), float(pixels.max())
    if hi <= lo:
        raise ValueError("Constant image")
    # Fixed per-image operation: no learned statistics from validation or test.
    scaled = np.rint(255 * (pixels - lo) / (hi - lo)).astype(np.uint8)
    im = Image.fromarray(scaled)
    im = ImageOps.pad(im, (size, size), method=Image.Resampling.BILINEAR,
                      color=0, centering=(0.5, 0.5))
    return im, {"source_height": pixels.shape[0], "source_width": pixels.shape[1],
                "source_min": lo, "source_max": hi}


def prepare_data(manifest, output_dir, cfg):
    """Create immutable data audit and model inputs. Do not silently drop failures."""
    cfg.validate()
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=False)
    write_json(output_dir / "status.json", {"created_utc": utc_now(), "status": "preparing"})
    df = make_split(read_manifest(manifest), cfg.split_seed)
    images = output_dir / "images"
    images.mkdir()
    rows, failures = [], []
    for row in df.to_dict("records"):
        try:
            im, info = prepare_image(row["image_path"], cfg.image_size)
            dest = (images / f'{row["image_id"]}.png').resolve()
            im.save(dest)
            row.update(info)
            row["prepared_path"] = str(dest)
            row["raw_hash"] = sha256(row["image_path"])
            row["pixel_hash"] = hashlib.sha256(im.tobytes()).hexdigest()
            rows.append(row)
        except Exception as e:
            failures.append({"image_id": row["image_id"], "error": str(e)})
    pd.DataFrame(failures, columns=["image_id", "error"]).to_csv(
        output_dir / "conversion_failures.csv", index=False)
    if failures:
        raise ValueError(f"{len(failures)} conversion failures. See conversion_failures.csv")
    result = pd.DataFrame(rows)
    result.to_csv(output_dir / "audit.csv", index=False)
    validate_separation(result)
    counts = result.groupby(["split", "label"]).agg(
        images=("image_id", "count"), patients=("patient_id", "nunique"))
    counts.to_csv(output_dir / "counts.csv")
    for split, part in result.groupby("split"):
        part.to_csv(output_dir / f"{split}.csv", index=False)
    write_json(output_dir / "status.json", {"created_utc": utc_now(), "status": "prepared",
        "manifest_sha256": sha256(manifest), "split_seed": cfg.split_seed,
        "image_size": cfg.image_size, "data_verified": False,
        "preprocessing": "grayscale; per-image minmax; 8-bit; aspect-preserving pad"})
    return result


def verify_data(data_dir, reviewer_note):
    if not str(reviewer_note).strip():
        raise ValueError("Describe which images and metadata you checked")
    path = Path(data_dir) / "status.json"
    status = json.loads(path.read_text())
    if status.get("status") != "prepared":
        raise ValueError("Data preparation has not completed")
    status.update(data_verified=True, verification_utc=utc_now(),
                  reviewer_note=reviewer_note)
    write_json(path, status)


def tf_module(seed):
    import tensorflow as tf
    tf.keras.utils.set_random_seed(seed)
    tf.config.experimental.enable_op_determinism()
    return tf


def image_dataset(df, cfg, training=False, vgg=False):
    import tensorflow as tf
    ds = tf.data.Dataset.from_tensor_slices(
        (df.prepared_path.to_numpy(), df.label.to_numpy(dtype=np.float32)))
    if training:
        ds = ds.shuffle(len(df), seed=cfg.train_seed, reshuffle_each_iteration=True)
    def decode(path, y):
        x = tf.io.decode_png(tf.io.read_file(path), channels=3)
        x = tf.cast(x, tf.float32)
        x.set_shape((cfg.image_size, cfg.image_size, 3))
        if vgg:
            x = tf.keras.applications.vgg16.preprocess_input(x)
        return x, tf.reshape(y, (1,))
    return ds.map(decode, num_parallel_calls=tf.data.AUTOTUNE).batch(
        cfg.batch_size).prefetch(tf.data.AUTOTUNE)


def build_cnn(cfg):
    import tensorflow as tf
    L = tf.keras.layers
    inputs = tf.keras.Input((cfg.image_size, cfg.image_size, 3))
    x = L.Rescaling(1.0 / 255)(inputs)
    x = L.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = L.MaxPooling2D()(x)
    x = L.Conv2D(64, 3, padding="same", activation="relu", name="last_conv")(x)
    x = L.MaxPooling2D()(x)
    x = L.GlobalAveragePooling2D()(x)
    x = L.Dense(128, activation="relu")(x)
    x = L.Dropout(0.5)(x)
    return tf.keras.Model(inputs, L.Dense(1, activation="sigmoid")(x), name="scratch_cnn")


def build_vgg(cfg, weights="imagenet"):
    """The training pipeline ALWAYS calls this with ImageNet weights."""
    import tensorflow as tf
    base = tf.keras.applications.VGG16(include_top=False, weights=weights,
        input_shape=(cfg.image_size, cfg.image_size, 3), pooling="avg")
    base.trainable = False
    head_input = tf.keras.Input((512,))
    x = tf.keras.layers.Dense(128, activation="relu")(head_input)
    x = tf.keras.layers.Dropout(0.5)(x)
    head = tf.keras.Model(head_input, tf.keras.layers.Dense(1, activation="sigmoid")(x),
                          name="vgg_classifier")
    return base, head


def new_run(data_dir, runs_dir, cfg):
    cfg.validate()
    data_dir = Path(data_dir).resolve()
    data_status = json.loads((data_dir / "status.json").read_text())
    if not data_status.get("data_verified"):
        raise ValueError("Inspect images and metadata, then run verify_data first")
    if data_status['image_size'] != cfg.image_size or data_status['split_seed'] != cfg.split_seed:
        raise ValueError("Config does not match the prepared dataset")
    run = Path(runs_dir).resolve() / cfg.experiment_id
    run.mkdir(parents=True, exist_ok=False)
    write_json(run / "config.json", asdict(cfg))
    source_path = Path(__file__).resolve()
    (run / "project_source.py").write_bytes(source_path.read_bytes())
    split_hashes = {s: sha256(data_dir / f"{s}.csv") for s in ["train", "val", "test"]}
    write_json(run / "status.json", {"created_utc": utc_now(), "stage": "created",
        "data_dir": str(data_dir), "data_status": data_status,
        "split_hashes": split_hashes, "source_sha256": sha256(source_path),
        "dataset": "user-supplied CBIS-DDSM mass crops"})
    versions = {}
    for pkg in ["tensorflow", "tensorflow-cpu", "keras", "numpy", "pandas",
                "scikit-learn", "pydicom", "Pillow", "matplotlib"]:
        try:
            versions[pkg] = importlib.metadata.version(pkg)
        except importlib.metadata.PackageNotFoundError:
            versions[pkg] = "not installed under this package name"
    write_json(run / "environment.json", {"python": platform.python_version(),
        "platform": platform.platform(), "packages": versions})
    freeze = sorted(f"{d.metadata['Name']}=={d.version}" for d in importlib.metadata.distributions()
                    if d.metadata.get("Name"))
    (run / "environment_freeze.txt").write_text('\n'.join(freeze), encoding="utf-8")
    return run


def load_run(run):
    run = Path(run)
    cfg = Config(**json.loads((run / "config.json").read_text()))
    status = json.loads((run / "status.json").read_text())
    for split, digest in status['split_hashes'].items():
        if sha256(Path(status['data_dir']) / f'{split}.csv') != digest:
            raise ValueError("Split file changed since run creation")
    if sha256(Path(__file__)) != status['source_sha256']:
        raise ValueError("Source code changed since run creation; use saved source or a new run")
    return cfg, status


def train_models(run):
    run = Path(run)
    cfg, status = load_run(run)
    if status['stage'] != 'created':
        raise ValueError("This run has already started; use a new experiment ID for a new run")
    tf = tf_module(cfg.train_seed)
    status.update(stage="training", training_started_utc=utc_now())
    write_json(run / "status.json", status)
    train = pd.read_csv(Path(status['data_dir']) / 'train.csv')
    val = pd.read_csv(Path(status['data_dir']) / 'val.csv')
    weights = compute_class_weight("balanced", classes=np.array([0, 1]), y=train.label)
    class_weights = {int(k): float(v) for k, v in enumerate(weights)}
    write_json(run / 'class_weights.json', class_weights)
    write_json(run / 'devices.json', {"devices": [str(d) for d in tf.config.list_physical_devices()]})
    timings = []
    for name in ['cnn', 'vgg16']:
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(cfg.train_seed)
        folder = run / name
        folder.mkdir()
        model_start = time.perf_counter()
        started = utc_now()
        if name == 'cnn':
            model = build_cnn(cfg)
            train_ds = image_dataset(train, cfg, training=True)
            val_ds = image_dataset(val, cfg)
            feature_seconds = 0.0
        else:
            base, model = build_vgg(cfg, weights='imagenet')
            feature_start = time.perf_counter()
            train_features = base.predict(image_dataset(train, cfg, vgg=True), verbose=1)
            val_features = base.predict(image_dataset(val, cfg, vgg=True), verbose=1)
            feature_seconds = time.perf_counter() - feature_start
            # Cache only train/validation features. Test is evaluated later.
            np.savez_compressed(folder / 'development_features.npz',
                train=train_features, val=val_features,
                train_ids=train.image_id.to_numpy(dtype=str),
                val_ids=val.image_id.to_numpy(dtype=str))
            train_ds = tf.data.Dataset.from_tensor_slices(
                (train_features, train.label.to_numpy(dtype=np.float32).reshape(-1, 1)))
            train_ds = train_ds.shuffle(len(train), seed=cfg.train_seed).batch(cfg.batch_size)
            val_ds = tf.data.Dataset.from_tensor_slices(
                (val_features, val.label.to_numpy(dtype=np.float32).reshape(-1, 1))).batch(cfg.batch_size)
        model.compile(optimizer=tf.keras.optimizers.Adam(cfg.learning_rate),
            loss='binary_crossentropy', metrics=[tf.keras.metrics.BinaryAccuracy(name='accuracy'),
            tf.keras.metrics.AUC(name='auc'), tf.keras.metrics.Recall(name='recall')])
        summary = []
        model.summary(print_fn=lambda line: summary.append(line))
        (folder / 'trained_model_summary.txt').write_text('\n'.join(summary), encoding='utf-8')
        class EpochClock(tf.keras.callbacks.Callback):
            def on_epoch_begin(self, epoch, logs=None):
                self.start = time.perf_counter()
            def on_epoch_end(self, epoch, logs=None):
                # Keep string timestamps out of Keras numeric progress metrics.
                row = dict(epoch=epoch, **(logs or {}),
                           epoch_seconds=time.perf_counter() - self.start,
                           ended_utc=utc_now())
                path = folder / 'training_log.csv'
                exists = path.exists()
                with open(path, 'a', newline='', encoding='utf-8') as stream:
                    writer = csv.DictWriter(stream, fieldnames=list(row))
                    if not exists:
                        writer.writeheader()
                    writer.writerow(row)
        callbacks = [EpochClock(),
            tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=cfg.patience,
                restore_best_weights=True),
            tf.keras.callbacks.ModelCheckpoint(str(folder / 'best.weights.h5'),
                monitor='val_loss', save_best_only=True, save_weights_only=True),
            tf.keras.callbacks.TerminateOnNaN()]
        fit_start = time.perf_counter()
        history = model.fit(train_ds, validation_data=val_ds, epochs=cfg.epochs,
                            class_weight=class_weights, callbacks=callbacks, verbose=1)
        fit_seconds = time.perf_counter() - fit_start
        values = np.asarray(history.history['val_loss'], dtype=float)
        if not np.isfinite(values).all():
            raise ValueError('Nonfinite validation loss; do not evaluate failed training')
        model.load_weights(folder / 'best.weights.h5')
        if name == 'vgg16':
            inputs = tf.keras.Input((cfg.image_size, cfg.image_size, 3))
            full_model = tf.keras.Model(inputs, model(base(inputs, training=False)),
                                        name='frozen_imagenet_vgg16')
        else:
            full_model = model
        full_model.save(folder / 'model.keras')
        summary = []
        full_model.summary(print_fn=lambda line: summary.append(line), expand_nested=True)
        (folder / 'full_model_summary.txt').write_text('\n'.join(summary), encoding='utf-8')
        timings.append({"model": name, "started_utc": started, "ended_utc": utc_now(),
            "epochs": len(values), "best_epoch": int(np.argmin(values)) + 1,
            "feature_seconds": feature_seconds, "fit_seconds": fit_seconds,
            "total_seconds": time.perf_counter() - model_start,
            "total_parameters": full_model.count_params(),
            "trainable_parameters": int(sum(np.prod(w.shape) for w in full_model.trainable_weights)),
            "pretrained_weights": 'ImageNet' if name == 'vgg16' else 'none',
            "model_sha256": sha256(folder / 'model.keras')})
        pd.DataFrame(timings).to_csv(run / 'training_summary.csv', index=False)
    status.update(stage='trained', training_completed_utc=utc_now())
    write_json(run / 'status.json', status)
    return pd.DataFrame(timings)


def metric_values(y, score, threshold=0.5):
    y, score = np.asarray(y, dtype=int), np.asarray(score, dtype=float)
    if not np.isfinite(score).all() or ((score < 0) | (score > 1)).any():
        raise ValueError('Scores must be finite values in [0,1]')
    pred = (score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    both = len(np.unique(y)) == 2
    return {"accuracy": accuracy_score(y, pred),
        "precision": precision_score(y, pred, zero_division=0),
        "sensitivity": recall_score(y, pred, zero_division=0),
        "specificity": float(tn / (tn + fp)) if tn + fp else None,
        "f1": f1_score(y, pred, zero_division=0),
        "roc_auc": roc_auc_score(y, score) if both else None,
        "average_precision": average_precision_score(y, score) if both else None,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}


def paired_bootstrap(predictions, repeats=1000, seed=42):
    """Paired patient-cluster bootstrap of image-level ROC-AUC and sensitivity."""
    grouped = [np.asarray(indices) for indices in predictions.groupby('patient_id').indices.values()]
    if not grouped:
        raise ValueError('No patients')
    rng = np.random.default_rng(seed)
    samples = {f'{m}_{metric}': [] for m in ['cnn', 'vgg16', 'difference']
               for metric in ['roc_auc', 'sensitivity']}
    for _ in range(repeats):
        selected = rng.integers(0, len(grouped), len(grouped))
        idx = np.concatenate([grouped[i] for i in selected])
        b = predictions.iloc[idx]
        if b.label.nunique() < 2:
            continue
        scores = {m: metric_values(b.label, b[f'{m}_score']) for m in ['cnn', 'vgg16']}
        for metric in ['roc_auc', 'sensitivity']:
            for m in ['cnn', 'vgg16']:
                samples[f'{m}_{metric}'].append(scores[m][metric])
            samples[f'difference_{metric}'].append(scores['vgg16'][metric] - scores['cnn'][metric])
    rows = []
    for key, values in samples.items():
        lower, upper = np.quantile(values, [0.025, 0.975]) if values else [None, None]
        rows.append({'quantity': key, 'lower_95': lower, 'upper_95': upper,
                     'valid_replicates': len(values), 'requested_replicates': repeats})
    return pd.DataFrame(rows)


def evaluate_models(run):
    run = Path(run)
    cfg, status = load_run(run)
    if status['stage'] != 'trained':
        raise ValueError('Evaluate only a fully trained, unevaluated run')
    destination = run / 'evaluation'
    destination.mkdir(exist_ok=False)
    tf = tf_module(cfg.train_seed)
    test = pd.read_csv(Path(status['data_dir']) / 'test.csv')
    train = pd.read_csv(Path(status['data_dir']) / 'train.csv')
    pred = test[['image_id', 'patient_id', 'label']].copy()
    for optional in ['breast_density', 'image_view']:
        if optional in test:
            pred[optional] = test[optional]
    timings = pd.read_csv(run / 'training_summary.csv').set_index('model')
    metrics = []
    for name in ['cnn', 'vgg16']:
        path = run / name / 'model.keras'
        if sha256(path) != timings.loc[name, 'model_sha256']:
            raise ValueError('Trained model file changed')
        model = tf.keras.models.load_model(path, compile=False)
        start = time.perf_counter()
        score = model.predict(image_dataset(test, cfg, vgg=(name == 'vgg16')), verbose=1).ravel()
        pred[f'{name}_score'] = score
        values = metric_values(test.label, score, cfg.threshold)
        metrics.append(dict(model=name, **values, threshold=cfg.threshold,
            images=len(test), patients=test.patient_id.nunique(),
            prediction_seconds=time.perf_counter()-start))
        del model
        tf.keras.backend.clear_session()
    majority = int(train.label.value_counts().idxmax())
    metrics.append(dict(model='training_majority_reference',
        **metric_values(test.label, np.full(len(test), majority)), threshold=cfg.threshold,
        images=len(test), patients=test.patient_id.nunique(), prediction_seconds=0))
    results = pd.DataFrame(metrics)
    results.to_csv(destination / 'measured_results.csv', index=False)
    pred.to_csv(destination / 'test_predictions.csv', index=False)
    paired_bootstrap(pred, cfg.bootstrap_repetitions, cfg.train_seed).to_csv(
        destination / 'paired_patient_bootstrap.csv', index=False)
    # Exploratory slices: report sample sizes and avoid claiming demographic fairness.
    slices = []
    for column in ['breast_density', 'image_view']:
        if column not in pred:
            continue
        for group, frame in pred.groupby(column, dropna=False):
            for name in ['cnn', 'vgg16']:
                slices.append(dict(group_column=column, group=str(group), model=name,
                    images=len(frame), patients=frame.patient_id.nunique(),
                    malignant_images=int(frame.label.sum()),
                    **metric_values(frame.label, frame[f'{name}_score'])))
    if slices:
        pd.DataFrame(slices).to_csv(destination / 'exploratory_subgroups.csv', index=False)
    status.update(stage='evaluated', evaluation_completed_utc=utc_now())
    write_json(run / 'status.json', status)
    make_figures(run)
    make_report(run)
    return results


def make_figures(run):
    import matplotlib.pyplot as plt
    from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay
    run = Path(run)
    ev = run / 'evaluation'
    for name in ['cnn', 'vgg16']:
        hist = pd.read_csv(run / name / 'training_log.csv')
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        for ax, metric in zip(axes, ['loss', 'accuracy']):
            ax.plot(hist.epoch + 1, hist[metric], label='Training')
            ax.plot(hist.epoch + 1, hist[f'val_{metric}'], label='Validation')
            ax.set(xlabel='Epoch', ylabel=metric, title=f'{name}: {metric}')
            ax.legend()
        fig.tight_layout(); fig.savefig(ev / f'{name}_learning_curves.png', dpi=160); plt.close(fig)
    pred = pd.read_csv(ev / 'test_predictions.csv')
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, name in zip(axes, ['cnn', 'vgg16']):
        ConfusionMatrixDisplay.from_predictions(pred.label, (pred[f'{name}_score'] >= .5).astype(int),
            display_labels=['Benign', 'Malignant'], ax=ax, colorbar=False, cmap='Blues')
        ax.set_title(name)
    fig.tight_layout(); fig.savefig(ev / 'confusion_matrices.png', dpi=160); plt.close(fig)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for name in ['cnn', 'vgg16']:
        RocCurveDisplay.from_predictions(pred.label, pred[f'{name}_score'], name=name, ax=axes[0])
        PrecisionRecallDisplay.from_predictions(pred.label, pred[f'{name}_score'], name=name, ax=axes[1])
    axes[1].axhline(pred.label.mean(), linestyle='--', color='gray', label='Test prevalence')
    axes[1].legend(); fig.tight_layout()
    fig.savefig(ev / 'roc_and_precision_recall.png', dpi=160); plt.close(fig)


def make_report(run):
    run = Path(run)
    results = pd.read_csv(run / 'evaluation/measured_results.csv')
    times = pd.read_csv(run / 'training_summary.csv')
    pred = pd.read_csv(run / 'evaluation/test_predictions.csv')
    lines = ['# Measured experiment results', '', f'Generated UTC: {utc_now()}', '',
        'These values were calculated from the saved test predictions for this run.',
        'Prediction unit: lesion image. Splits are patient-separated; scores are not patient-level screening results.', '',
        '| Model | ROC AUC | Sensitivity | Specificity | F1 |', '| --- | --- | --- | --- | --- |']
    for r in results.to_dict('records'):
        lines.append(f"| {r['model']} | {r['roc_auc']:.4f} | {r['sensitivity']:.4f} | {r['specificity']:.4f} | {r['f1']:.4f} |")
    lines.extend(['', f'Test images: {len(pred)}. Unique test patients: {pred.patient_id.nunique()}.',
        f'Malignant images: {int(pred.label.sum())}; benign images: {int((pred.label == 0).sum())}.', '',
        'See paired_patient_bootstrap.csv for paired patient-cluster percentile intervals.',
        'These intervals describe test-sample uncertainty conditional on these fitted models; they do not measure training-seed uncertainty.', '',
        '## Training evidence', ''])
    for r in times.to_dict('records'):
        lines.append(f"- {r['model']}: {r['epochs']} epochs; best epoch {r['best_epoch']}; total {r['total_seconds']:.1f} seconds; trainable parameters {r['trainable_parameters']}.")
    lines.extend(['', '## Interpretation still to write', '',
        'Discuss false negatives and false positives, uncertainty, class imbalance, source-image conversion and the restricted dataset.',
        'Do not claim that the larger point estimate proves superiority or that this prototype is clinically validated.',
        'Stakeholder feedback is recorded separately; no responses are inferred from model scores.', ''])
    (run / 'evaluation/RESULTS.md').write_text('\n'.join(lines), encoding='utf-8')


def predict_image(run, model_name, path):
    if model_name not in {'cnn', 'vgg16'}:
        raise ValueError('model_name must be cnn or vgg16')
    cfg, _ = load_run(run)
    tf = tf_module(cfg.train_seed)
    model = tf.keras.models.load_model(Path(run) / model_name / 'model.keras', compile=False)
    image, _ = prepare_image(path, cfg.image_size)
    x = np.repeat(np.asarray(image)[..., None], 3, axis=-1)[None].astype(np.float32)
    if model_name == 'vgg16':
        x = tf.keras.applications.vgg16.preprocess_input(x)
    score = float(model.predict(x, verbose=0)[0, 0])
    return {'model': model_name, 'predicted_class': 'Malignant' if score >= cfg.threshold else 'Benign',
            'malignant_score': score, 'threshold': cfg.threshold,
            'note': 'Research model score; not a calibrated clinical probability.'}


def summarise_feedback(csv_path, destination):
    df = pd.read_csv(csv_path, keep_default_na=False)
    destination = Path(destination)
    if df.empty:
        text = '# Stakeholder feedback status\n\nNo stakeholder responses have been collected.\n'
    else:
        if df.participant_id.duplicated().any() or (df.participant_id == '').any():
            raise ValueError('Use one anonymous ID per respondent')
        if not df.consent_to_use.astype(str).str.lower().isin(['yes']).all():
            raise ValueError('Resolve consent before including these responses')
        dates = pd.to_datetime(df['date_utc'], utc=True, errors='raise')
        if (dates > pd.Timestamp.now(tz='UTC')).any():
            raise ValueError('Response dates cannot be in the future')
        text = f'# Stakeholder feedback summary\n\nActual responses: {len(df)}.\n\n'
        for col in ['clarity', 'ease_of_use', 'limitations_understood']:
            values = pd.to_numeric(df[col], errors='raise')
            if not values.between(1, 5).all():
                raise ValueError('Ratings must be 1 to 5')
            text += f'- {col}: median {values.median():.1f}; range {values.min()} to {values.max()}; n={len(values)}.\n'
        text += '\nRoles represented: ' + ', '.join(sorted(set(df.role))) + '.\n'
        text += '\nReview free-text responses manually for themes; record resulting changes in change_log.csv.\n'
        text += 'Convenience feedback does not validate diagnostic accuracy. Non-clinicians cannot establish clinical usefulness.\n'
    destination.write_text(text, encoding='utf-8')
    return text

```


### Appendix B — prepare_cbis.py (explicit-override mapping adapter)

```python
"""Map official CBIS-DDSM mass metadata to local crop files, without guessing.

Exact paths are tried first, then unique SeriesInstanceUID matches. Ambiguous
series require an explicit override. No patient-path substring matching is used.
"""
from pathlib import Path, PurePosixPath
import re
import pandas as pd
from project import canonical_patient


def build_manifest(train_csv, test_csv, image_root, destination, overrides=None):
    image_root, destination = Path(image_root).resolve(), Path(destination).resolve()
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        raise FileExistsError('Manifest already exists; preserve it or choose a new output name')
    if not image_root.is_dir():
        raise FileNotFoundError(f'Image root is not a folder: {image_root}')
    lookup = {}
    if overrides is not None:
        override_path = Path(overrides).resolve()
        override_df = pd.read_csv(override_path, dtype=str, keep_default_na=False)
        if override_df.source_path.duplicated().any():
            raise ValueError('Duplicate source_path in overrides')
        for r in override_df.itertuples(index=False):
            p = Path(r.local_path)
            lookup[r.source_path.strip().replace('\\', '/')] = (
                p if p.is_absolute() else override_path.parent / p).resolve()
    rows = []
    required = {'patient_id', 'pathology', 'cropped image file path'}
    for split, path in [('train', train_csv), ('test', test_csv)]:
        df = pd.read_csv(path, dtype=str, keep_default_na=False)
        df.columns = df.columns.str.strip()
        if not required <= set(df.columns):
            raise ValueError(f'{path} is missing official mass metadata columns')
        for _, r in df.iterrows():
            source = r['cropped image file path'].strip().replace('\\', '/')
            if not source:
                raise ValueError('Blank cropped image file path; inspect source CSV')
            row = {'patient_id': canonical_patient(r['patient_id']), 'pathology': r['pathology'],
                   'official_split': split, 'source_path': source,
                   'breast_density': r.get('breast_density', r.get('breast density', '')),
                   'image_view': r.get('image view', ''),
                   'lesion_id': r.get('abnormality id', '')}
            candidate = lookup.get(source, image_root / Path(source))
            row['image_path'] = str(candidate.resolve()) if candidate.is_file() else ''
            rows.append(row)
    # Header indexing is required only for files whose original hierarchy changed.
    unresolved = [r for r in rows if not r['image_path']]
    series_index, unreadable = {}, []
    if unresolved:
        import pydicom
        paths = sorted(p for p in image_root.rglob('*') if p.suffix.lower() in {'.dcm', '.dicom'})
        print(f'Indexing {len(paths)} DICOM headers; this may take a few minutes.')
        for p in paths:
            try:
                ds = pydicom.dcmread(p, stop_before_pixels=True,
                                    specific_tags=['SeriesInstanceUID'])
                uid = str(getattr(ds, 'SeriesInstanceUID', ''))
                if uid:
                    series_index.setdefault(uid, []).append(str(p.resolve()))
            except Exception as exc:
                unreadable.append({'path': str(p), 'error': str(exc)})
    failures = []
    for row in rows:
        if row['image_path']:
            continue
        parts = PurePosixPath(row['source_path']).parts
        uid = parts[-2] if len(parts) >= 2 else ''
        candidates = series_index.get(uid, []) if re.fullmatch(r'[0-9.]+', uid) else []
        if len(candidates) == 1:
            row['image_path'] = candidates[0]
        else:
            failures.append({'source_path': row['source_path'], 'local_path': '',
                'reason': 'ambiguous series' if candidates else 'no exact path or unique series match',
                'candidates': ' | '.join(candidates)})
    pd.DataFrame(unreadable, columns=['path', 'error']).to_csv(
        destination.parent / 'unreadable_dicom_headers.csv', index=False)
    if failures:
        failure_path = destination.parent / 'path_mapping_needed.csv'
        pd.DataFrame(failures).to_csv(failure_path, index=False)
        raise ValueError(f'{len(failures)} mappings unresolved. Fill local_path in {failure_path}, '
                         'then pass it as overrides. Choose the lesion IMAGE, not its ROI mask.')
    result = pd.DataFrame(rows)
    result.to_csv(destination, index=False)
    print(f'Saved {len(result)} manifest rows to {destination}')
    return result

```


### Appendix C — notebook evidence-check helper

```python
def check_experiment_evidence(run, verify_pixels=True):
    """Check local saved evidence without retraining or rewriting results."""
    import hashlib
    run = Path(run)
    saved_cfg, status = P.load_run(run)
    parts = {s: pd.read_csv(Path(status["data_dir"]) / f"{s}.csv")
             for s in ["train", "val", "test"]}
    for name, frame in parts.items():
        if frame.empty or set(frame["split"]) != {name}:
            raise ValueError(f"Invalid split contents: {name}")
        if set(frame["label"]) != {0, 1}:
            raise ValueError(f"Both classes are required in {name}")
    joined = pd.concat(parts.values(), ignore_index=True)
    if joined.image_id.duplicated().any():
        raise ValueError("An image ID appears more than once across split files")
    P.validate_separation(joined)
    if verify_pixels:
        for row in joined.itertuples(index=False):
            with Image.open(row.prepared_path) as im:
                if im.mode != "L" or im.size != (saved_cfg.image_size, saved_cfg.image_size):
                    raise ValueError(f"Unexpected prepared image format: {row.image_id}")
                digest = hashlib.sha256(im.tobytes()).hexdigest()
            if digest != row.pixel_hash:
                raise ValueError(f"Prepared pixels changed: {row.image_id}")
    if status["stage"] in {"trained", "evaluated"}:
        timing = pd.read_csv(run / "training_summary.csv").set_index("model")
        if set(timing.index) != {"cnn", "vgg16"} or not timing.index.is_unique:
            raise ValueError("Need one training summary per model")
        for name in ["cnn", "vgg16"]:
            if P.sha256(run / name / "model.keras") != timing.loc[name, "model_sha256"]:
                raise ValueError(f"Saved model changed: {name}")
            log = pd.read_csv(run / name / "training_log.csv")
            if len(log) != int(timing.loc[name, "epochs"]):
                raise ValueError(f"Epoch count mismatch: {name}")
            if not np.array_equal(log.epoch.to_numpy(), np.arange(len(log))):
                raise ValueError(f"Nonconsecutive epoch log: {name}")
            if not np.isfinite(log[["loss", "val_loss"]].to_numpy()).all():
                raise ValueError(f"Nonfinite losses: {name}")
            if int(np.argmin(log.val_loss.to_numpy())) + 1 != int(timing.loc[name, "best_epoch"]):
                raise ValueError(f"Best checkpoint epoch mismatch: {name}")
    if status["stage"] == "evaluated":
        pred = pd.read_csv(run / "evaluation/test_predictions.csv")
        test = parts["test"]
        if pred.image_id.duplicated().any() or set(pred.image_id) != set(test.image_id):
            raise ValueError("Predictions do not cover the saved test split exactly")
        aligned = pred.set_index("image_id").loc[test.image_id]
        if not np.array_equal(aligned.label.to_numpy(), test.label.to_numpy()):
            raise ValueError("Prediction labels differ from test labels")
        if not np.array_equal(aligned.patient_id.to_numpy(), test.patient_id.to_numpy()):
            raise ValueError("Prediction patient IDs differ from the test split")
        measured = pd.read_csv(run / "evaluation/measured_results.csv").set_index("model")
        expected = {"cnn", "vgg16", "training_majority_reference"}
        if set(measured.index) != expected or not measured.index.is_unique:
            raise ValueError("Unexpected measured-results rows")
        majority = int(parts["train"].label.value_counts().idxmax())
        for name in sorted(expected):
            score = (np.full(len(test), majority) if name == "training_majority_reference"
                     else aligned[f"{name}_score"].to_numpy())
            recalculated = P.metric_values(test.label, score, saved_cfg.threshold)
            for metric, value in recalculated.items():
                if not np.isclose(float(measured.loc[name, metric]), value, rtol=1e-6, atol=1e-8):
                    raise ValueError(f"Metric mismatch: {name}, {metric}")
            for field, value in [("images", len(test)), ("patients", test.patient_id.nunique()),
                                 ("threshold", saved_cfg.threshold)]:
                if not np.isclose(float(measured.loc[name, field]), value):
                    raise ValueError(f"Result metadata mismatch: {name}, {field}")
        print("Saved test metrics agree with the full prediction CSV.")
    print("Evidence checks passed for stage:", status["stage"])
    return joined.groupby("split").agg(images=("image_id", "size"),
                                      patients=("patient_id", "nunique"))
```


## References

- [CBIS-DDSM dataset paper](https://pmc.ncbi.nlm.nih.gov/articles/PMC5735920/)
- [TensorFlow classification tutorial](https://www.tensorflow.org/tutorials/images/classification)
- [Keras VGG16 preprocessing and model](https://keras.io/2/api/applications/vgg/)
- [Patient grouping with StratifiedGroupKFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedGroupKFold.html)
- [Pydicom pixel processing](https://pydicom.github.io/pydicom/stable/guides/user/working_with_pixel_data.html)

The primary protocol is an original project design informed by these technical references, not a reproduction of their reported mammography results.